In [ ]:
import os
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import signal
from glob import glob
from mne_bids import BIDSPath, read_raw_bids
from scipy.stats import ttest_rel
import math
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import pingouin as pg
%matplotlib qt

### define inputs

In [ ]:
# file paths
bids_root = '/Users/xjf5193/Library/CloudStorage/OneDrive-NorthwesternUniversity/SoundBrain Lab - EAM1/data-bids'

# derivatives from status
deriv_dir_motor = os.path.join(bids_root, 'derivatives', 'epochs_erp_motor')
if not os.path.exists(deriv_dir_motor):
    os.makedirs(deriv_dir_motor)

deriv_dir_status = os.path.join(bids_root, 'derivatives', 'epochs_erp_status-adjusted')
if not os.path.exists(deriv_dir_status):
    os.makedirs(deriv_dir_status) 

# Preprocess data (BIDS)

In [ ]:
task_list = ['active', 'passive', 'motor']
bl_start = 0.100
l_freq = 1
h_freq = 30
passive_hex = '#4477AA'
active_hex = '#CC6677'

### Cortically filtered (1–30 Hz) epochs

In [ ]:
# generate epochs for all participants and tasks, and save them in the derivatives folder
for sub_num in range(2, 35):
    sub_label = f'{sub_num:02d}'

    if sub_num == 8 or sub_num == 10:
        continue

    for task_label in task_list:
        print(f'Loading {task_label} data')

        epoch_list_ergo1 = []
        epoch_list_status = []

        for run_label in range(1,6):
            # load in EEG data
            bids_path = BIDSPath(root=bids_root,
                                 datatype='eeg',
                                 subject=str(sub_label), 
                                 task=task_label, 
                                 run=run_label)
            
            try:
                data = read_raw_bids(bids_path, verbose=False)
                data.load_data(verbose=False)

                # re-reference data to linked mastoid reference
                data_ref = data.set_eeg_reference(ref_channels=['M1', 'M2'], verbose=False)
                data_filtered = data_ref.copy().filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
                
                # Load events from the status channel
                events_path = bids_path.copy().update(suffix='events', extension='.tsv')
                events_df_status = pd.read_csv(events_path.fpath, delimiter='\t')

                # adjusting onset times for status events to account for the delay in the status channel
                if sub_num < 12:
                    events_df_status['onset'] = events_df_status['onset'] + 0.062
                else:
                    events_df_status['onset'] = events_df_status['onset'] + 0.1

                # (optional) dropping events that are start less than 300ms from the previous
                # diffs_onset = np.diff(events_df['onset'])
                # too_close = np.where(diffs_onset < 0.3)[0] + 1
                # events_df = events_df.drop(index=too_close).reset_index(drop=True)

                annot = mne.Annotations(onset=events_df_status.onset, duration=0.170, description=events_df_status.trial_type)
                data_filtered.set_annotations(annot)
                events_from_annot_status, event_dict2_status = mne.events_from_annotations(data_filtered)
                
                epoch_kwargs = dict(on_missing='warn',
                                    picks=['Cz'],
                                    tmin=-1*bl_start, 
                                    tmax=0.4,
                                    baseline=[-1*bl_start, 0],
                                    reject=dict(eeg=75e-6),
                                    verbose=False)
                
                status_epochs = mne.Epochs(data_filtered, 
                                    events=events_from_annot_status, 
                                    event_id=event_dict2_status, 
                                    **epoch_kwargs).drop_bad(verbose=False)
                epoch_list_status.append(status_epochs)

            except Exception as e:
                print(f"No run {run_label} for task-{task_label}: {e}")
        
        try:
            # combine epochs across runs
            all_epochs_status = mne.concatenate_epochs(epoch_list_status)
        except IndexError as e:
            print(f'cannot run for sub-{sub_label} task-{task_label} run-{run_label}: {e}')
        
        try:
            out_base = f'sub-{sub_label}_task-{task_label}_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'
            if task_label == 'motor':
                all_epochs_status.save(os.path.join(deriv_dir_motor, out_base), overwrite=True)
            else:
                all_epochs_status.save(os.path.join(deriv_dir_status, out_base), overwrite=True)
        except Exception as e:
            print(f"Error saving epochs for sub-{sub_label} task-{task_label}: {e}")
            continue


# Analyze whole-group data

Load status epochs - used for motor subtraction analysis.

In [ ]:
act_pos_avgs_st = []
act_neg_avgs_st = []
act_eps = sorted(glob(deriv_dir_status + f'/sub-*_task-active_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'))
for act_ep in act_eps:
    if f'sub-08' in act_ep or f'sub-10' in act_ep:
        continue
    try:
        sub_active_epochs = mne.read_epochs(act_ep, verbose='WARNING')
        act_neg_avgs_st.append(sub_active_epochs['active/neg'].average())
        act_pos_avgs_st.append(sub_active_epochs['active/pos'].average())
    except:
        pass

pas_pos_avgs_st = []
pas_neg_avgs_st = []
pas_eps = sorted(glob(deriv_dir_status + f'/sub-*_task-passive_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'))
for pas_ep in pas_eps:
    if f'sub-08' in pas_ep or f'sub-10' in pas_ep:
        continue
    try:
        sub_passive_epochs = mne.read_epochs(pas_ep, verbose='WARNING')
        pas_neg_avgs_st.append(sub_passive_epochs['passive/neg'].average())
        pas_pos_avgs_st.append(sub_passive_epochs['passive/pos'].average())
    except:
        pass

motor_avgs_st = []
motor_eps = sorted(glob(deriv_dir_motor + f'/sub-*_task-motor_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'))
for motor_ep in motor_eps:
    if f'sub-08' in motor_ep or f'sub-10' in motor_ep:
        continue
    try:
        sub_motor_epoch = mne.read_epochs(motor_ep, verbose=False)
        motor_avgs_st.append(sub_motor_epoch.average())
    except Exception as e:
        print(f"Error occurred: {e}")
        pass

Average polarities within each subject

In [ ]:
# status avgs
passive_avgs_st = [mne.combine_evoked([pas_pos_avgs_st[x], pas_neg_avgs_st[x]], weights='nave')
                for x in range(len(pas_pos_avgs_st))]
active_avgs_st = [mne.combine_evoked([act_pos_avgs_st[x], act_neg_avgs_st[x]], weights='nave')
                for x in range(len(act_pos_avgs_st))]

Grand averages

In [ ]:
# status grand averages
pas_pos_grandavg_st = mne.grand_average(pas_pos_avgs_st)
pas_neg_grandavg_st = mne.grand_average(pas_neg_avgs_st)
act_pos_grandavg_st = mne.grand_average(act_pos_avgs_st)
act_neg_grandavg_st = mne.grand_average(act_neg_avgs_st)

passive_grandavg_st = mne.grand_average(pas_pos_avgs_st+pas_neg_avgs_st)
active_grandavg_st = mne.grand_average(act_pos_avgs_st+act_neg_avgs_st)

motor_grandavg = mne.grand_average(motor_avgs_st)

### Motor subtraction

In [ ]:
# subtract the motor ERP from the active and passive ERPs to isolate the auditory processing component
subtracted = mne.combine_evoked([active_grandavg_st, motor_grandavg], weights=[1, -1])
added = mne.combine_evoked([passive_grandavg_st, motor_grandavg], weights=[1, 1])

evokeds = {
    'Active': active_grandavg_st,
    'Passive': passive_grandavg_st,
    'Active - Motor': subtracted,
    'Motor': motor_grandavg
}

# plot all (active, passive, motor, subtraction)
# mne.viz.plot_compare_evokeds(
#     evokeds,
#     picks='Cz',
#     legend=True
# )

In [ ]:
# create the subtracted waveform (by subject) -> so we can generate confidence intervals for the subtraction waveform
print(len(active_avgs_st), len(motor_avgs_st), len(passive_avgs_st))
subtracted = []
for i in range(len(active_avgs_st)):
    subtracted.append(mne.combine_evoked([active_avgs_st[i], motor_avgs_st[i]], weights=[1, -1]))

In [ ]:
# plot all status conditions with confidence intervals
evokeds = {"Active": active_avgs_st, "Passive": passive_avgs_st, "Motor": motor_avgs_st, "Active - Motor": subtracted}

### Peak latency and amplitude

In [ ]:
# calculate peak latency and amplitude for all status conditions (passive, active, subtraction)
f_low=4
f_high=30
t_low=0.00
t_high=0.3

# Extract latency and amplitude for passive condition
pas_peak_lat_list = []
pas_peak_amp_list = []
for evk in passive_avgs_st:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    pas_peak_lat_list.append(peak[1])
    pas_peak_amp_list.append(peak[2])

# Extract latency and amplitude for active condition
act_peak_lat_list = []
act_peak_amp_list = []
for evk in active_avgs_st:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    act_peak_lat_list.append(peak[1])
    act_peak_amp_list.append(peak[2])


# subtracted
mot_peak_lat_list = []
mot_peak_amp_list = []
for evk in subtracted:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    mot_peak_lat_list.append(peak[1])
    mot_peak_amp_list.append(peak[2])

# Create a DataFrame for power values
peak_lat_dict = {'passive': pas_peak_lat_list,  
                 'active': act_peak_lat_list,
                 'motor': mot_peak_lat_list}

peak_amp_dict = {'passive': pas_peak_amp_list,  
                 'active': act_peak_amp_list,
                 'motor': mot_peak_amp_list}
peak_lat_df = pd.DataFrame(peak_lat_dict)
peak_amp_df = pd.DataFrame(peak_amp_dict)

In [ ]:
# convert cohen's d to a metric bound, used in TOST
def d_to_raw_bound(x, y, d):
    diff = x - y
    sd_diff = np.std(diff, ddof=1)
    return d * sd_diff

In [ ]:
# latency stats
peak_pass_mean = peak_lat_df['passive'].mean()
peak_sub_mean = peak_lat_df['motor'].mean()
print(f'Mean passive peak latency: {peak_pass_mean:.03f} s')
print(f'Mean active - subtracted peak latency: {peak_sub_mean:.03f} s')
print(f'Mean active peak latency: {peak_lat_df["active"].mean():.03f} s')

# Perform paired t-test
print("=== Passive vs Motor-Corrected Active ===")
# this test turned out significant because the "peak" of the motor-corrected active waveform is corresponding to a different peak in the passive waveform, 
# so the latencies are not aligned. The corresponding peak in the motor-corrected active waveform is suppressed so it is not detected as the same peak in the passive waveform
t_stat, p_value = ttest_rel(peak_lat_df['passive'], peak_lat_df['motor'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")
# tost
equiv_range = d_to_raw_bound(peak_lat_df['passive'], peak_lat_df['motor'], d=0.5)
tost_result = pg.tost(peak_lat_df['passive'], peak_lat_df['motor'], paired=True, bound=equiv_range)
print(tost_result)
# bayes factor
bf_result = pg.ttest(peak_lat_df['passive'], peak_lat_df['motor'], paired=True)
print(bf_result)

print("=== Active vs Motor-Corrected Active ===")
t_stat, p_value = ttest_rel(peak_lat_df['active'], peak_lat_df['motor'])
print(f"active t-statistic: {t_stat:.03f}, active p-value: {p_value:.05f}")
# tost
equiv_range = d_to_raw_bound(peak_lat_df['active'], peak_lat_df['motor'], d=0.5)
tost_result = pg.tost(peak_lat_df['active'], peak_lat_df['motor'], paired=True, bound=equiv_range)
print(tost_result)
# bayes factor
bf_result = pg.ttest(peak_lat_df['active'], peak_lat_df['motor'], paired=True)
print(bf_result)

print("=== Active vs Passive ===")
t_stat, p_value = ttest_rel(peak_lat_df['active'], peak_lat_df['passive'])
print(f"active t-statistic: {t_stat:.03f}, active p-value: {p_value:.05f}")
# tost
equiv_range = d_to_raw_bound(peak_lat_df['active'], peak_lat_df['passive'], d=0.5)
tost_result = pg.tost(peak_lat_df['active'], peak_lat_df['passive'], paired=True, bound=equiv_range)
print(tost_result)
# bayes factor
bf_result = pg.ttest(peak_lat_df['active'], peak_lat_df['passive'], paired=True)
print(bf_result)

In [ ]:
# amplitude stats
peak_df = pd.DataFrame(peak_amp_dict)
print(peak_df.head())

# Compute the mean peak for each condition
peak_pass_mean = peak_df['passive'].mean()
peak_sub_mean = peak_df['motor'].mean()
print(f'Mean passive peak amplitude: {peak_pass_mean} ')
print(f'Mean active - subtracted peak amplitude: {peak_sub_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_df['passive'], peak_df['motor'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")
# tost
equiv_range = d_to_raw_bound(peak_df['passive'], peak_df['motor'], d=0.5)
tost_result = pg.tost(peak_df['passive'], peak_df['motor'], paired=True, bound=equiv_range)
print(tost_result)
# bayes factor
bf_result = pg.ttest(peak_df['passive'], peak_df['motor'], paired=True)
print(bf_result)

### Power stats

In [ ]:
def compute_power(avg_evoked, f_low=90, f_high=110, t_low=0.1, t_high=0.2):
    # Compute the spectrogram (time-frequency representation)
    power = mne.time_frequency.tfr_multitaper(avg_evoked, 
                                            freqs=np.arange(f_low, f_high, 1), 
                                            n_cycles=2, 
                                            time_bandwidth=4.0,
                                            average=True, 
                                            return_itc=False)

    # Select the time and frequency bands
    time_mask = (power.times >= t_low) & (power.times <= t_high)
    freq_mask = (power.freqs >= f_low) & (power.freqs <= f_high)

    # Extract the power within the specified bands
    selected_power = power.data[:, freq_mask, :][:, :, time_mask]

    # Average power over the selected time and frequency bands
    average_power = selected_power.mean(axis=2).mean(axis=1)[0]
    return average_power

In [ ]:
f_low=4
f_high=30
t_low=-0.04
t_high=0.1

# Extract power for passive condition
pas_power_list = []
for evk in passive_avgs_st:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    pas_power_list.append(power)

# Extract power for active condition
act_power_list = []
for evk in active_avgs_st:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    act_power_list.append(power)

# Create a DataFrame for power values
power_dict = {'passive': pas_power_list,  
              'active': act_power_list}
power_df = pd.DataFrame(power_dict)

In [ ]:
# Compute the mean power for each condition
power_pass_mean = power_df['passive'].mean()
power_act_mean = power_df['active'].mean()
print(f'Mean passive power: {power_pass_mean}')
print(f'Mean active power: {power_act_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(power_df['passive'], power_df['active'])
print(f"power t-statistic: {t_stat:.03f}, p-value: {p_value:.05f}")
# tost
equiv_range = d_to_raw_bound(power_df['passive'], power_df['active'], d=0.5)
tost_result = pg.tost(power_df['passive'], power_df['active'], paired=True, bound=equiv_range)
print(tost_result)
# bayes factor
bf_result = pg.ttest(power_df['passive'], power_df['active'], paired=True)
print(bf_result)

#### Compute grand average power

In [ ]:
# Extract power for each condition
f_low=4
f_high=15
t_low=0.1
t_high=0.2

passive_evoked_power = compute_power(passive_grandavg_st, 
                                     f_low=f_low, f_high=f_high, 
                                     t_low=t_low, t_high=t_high)

active_evoked_power = compute_power(active_grandavg_st, 
                                    f_low=f_low, f_high=f_high, 
                                    t_low=t_low, t_high=t_high)


### frequency domain (spectra and spectrograms)

In [ ]:
# Power spectra
passive_psd = passive_grandavg_st.compute_psd(tmin=0.0, tmax=0.300, fmin=1, fmax=30)
active_psd = active_grandavg_st.compute_psd(tmin=0.0, tmax=0.300, fmin=1, fmax=30)

# Spectrograms
frequencies = np.arange(4, 20)

passive_power = passive_grandavg_st.compute_tfr('multitaper', 
                                               freqs=frequencies, 
                                               n_cycles=1)
active_power = active_grandavg_st.compute_tfr('multitaper', 
                                             freqs=frequencies, 
                                             n_cycles=1)

### Plot Results

In [ ]:
# Manuscript figure 3: ERP waveforms, spectrograms
plt.close("all")
fig, axes = plt.subplot_mosaic(
    [["ERP", "ERP", "ERP"],
     ["diff_spectrogram", "a_spectrogram", "p_spectrogram"]],
     constrained_layout=True,
    #  width_ratios=[2, 2, 1, 1],
     dpi=150,
     figsize=(5, 7)
)

SMALL  = 11    # tick labels
MEDIUM = 12    # axis labels, colorbar label  
LARGE  = 13    # subplot titles

plt.rcParams.update({
    'font.size':        SMALL,
    'axes.titlesize':   LARGE,
    'axes.labelsize':   MEDIUM,
    'xtick.labelsize':  SMALL,
    'ytick.labelsize':  SMALL,
})

# set figure legend
# fig.get_layout_engine().set(rect=[0, 0, 1, 0.92], hspace=0.0, wspace=0.0)
# passive_patch = mpatches.Patch(color=passive_hex, label='Passive')
# active_patch  = mpatches.Patch(color=active_hex,  label='Active')

# fig.legend(
#     handles=[passive_patch, active_patch],
#     loc='upper center',
#     ncols=2,
#     frameon=False,
#     fontsize=10,
# )

# ERP waveform
mne.viz.plot_compare_evokeds(evokeds, 
                             truncate_xaxis=False, 
                             truncate_yaxis=False,
                             picks='Cz', 
                             ci=0.95,
                             axes=axes['ERP'],
                             colors={'Passive': passive_hex, 'Active - Motor': 'k'},
                             linestyles=dict(Passive='-', **{'Active - Motor': '--'}),
                             styles=dict(Active={"color": "#666666", 'alpha': 0.6}, Motor={"color": "#cccccc", 'alpha': 0.7}),
                             legend=False)
axes["ERP"].set_title("")
axes["ERP"].set_ylabel('Amplitude (µV)')
axes["ERP"].tick_params(direction="in")


# note i editted the location of this legend post-hoc in inkscape to make it fit better, but this is the code for the legend that goes with the motor subtraction plot
handle_passive  = mlines.Line2D([], [], color=passive_hex,  linewidth=1, label="Passive")
handle_sub     = mlines.Line2D([], [], color="black", linewidth=1, linestyle=":",  label="Active - Motor")
handle_active = mlines.Line2D([], [], color="#666666",  linewidth=1, label="Active")
handle_motor = mlines.Line2D([], [], color="#cccccc", linewidth=1, label="Motor")

axes["ERP"].legend(handles=[handle_passive, handle_sub, handle_active, handle_motor], loc="upper right", frameon=False, fontsize=7)

for collection in axes["ERP"].collections:
    collection.set_alpha(0.15)

diff_power = active_power.copy()
diff_power.data = active_power.data - passive_power.data

passive_power.plot(axes=axes["p_spectrogram"], 
                   tmax=0.4,
                   colorbar=False, 
                   show=False,
                   vlim=[0, 9e-9],
                   cmap='magma')
active_power.plot(axes=axes["a_spectrogram"], 
                  tmax=0.4,
                  colorbar=False, 
                  show=False,
                  vlim=[0, 9e-9],
                  cmap='magma')

diff_power.plot(axes=axes["diff_spectrogram"],
                tmax=0.4,
                colorbar=False,
                cmap='RdBu_r',  
                show=False)

axes["p_spectrogram"].set_title('Passive')
axes["a_spectrogram"].set_title('Active')
axes["p_spectrogram"].sharex(axes["diff_spectrogram"])
axes["a_spectrogram"].sharex(axes["diff_spectrogram"])
axes["p_spectrogram"].set_ylabel('')
axes["a_spectrogram"].set_ylabel('')
axes["p_spectrogram"].set_xlabel('Time (s)')
axes["a_spectrogram"].set_xlabel('Time (s)')
axes["p_spectrogram"].tick_params(direction="in")
axes["a_spectrogram"].tick_params(direction="in")
axes["diff_spectrogram"].set_title('Active - Passive')
axes["diff_spectrogram"].set_xlabel('Time (s)')
axes["diff_spectrogram"].set_ylabel('Frequency (Hz)')
axes["diff_spectrogram"].tick_params(direction="in")

# add a single shared colorbar between the two spectrograms
im = axes["p_spectrogram"].images[0]  # grab the image from one of the axes
cbar = fig.colorbar(im, ax=[axes["p_spectrogram"], axes["a_spectrogram"]], 
                    location='bottom', 
                    shrink=0.6,
                    label='Power ($\\times 10^{-9}$)')

im_diff = axes["diff_spectrogram"].images[0]
cbar_diff = fig.colorbar(im_diff, 
                         ax=axes["diff_spectrogram"], 
                         location='bottom',  
                           label='Power difference ($\\times 10^{-9}$)')

# Disconnect all MNE's interactive callbacks
fig.canvas.mpl_disconnect_all() if hasattr(fig.canvas, 'mpl_disconnect_all') else None
for cid in list(fig.canvas.callbacks.callbacks.get('scroll_event', {}).keys()):
    fig.canvas.mpl_disconnect(cid)

fig.savefig('cortical.svg', dpi=600)
